# 01 - Clinical Data Ingestion, Cleaning & Preprocessing Pipeline
**Sepsis Clinical Decision Support System (CDSS)**  
*Dataset: PhysioNet / Computing in Cardiology Challenge 2019*

---

### Pipeline Objectives:
1. **Raw Data Ingestion & Concatenation**: Consolidate individual patient `.psv` files or load existing `physionet_sepsis_combined.csv`.
2. **Exploratory Data Analysis (EDA)**: Profile missingness rates, feature distributions, and the severe class imbalance (~1.8% positive sepsis rate).
3. **Physiological Outlier Cleaning**: Enforce physiological validity ranges (e.g. clipping impossible HR, SBP, or Temp values to NaN).
4. **Patient-Level Longitudinal Imputation**: Perform patient-grouped Forward Fill (`ffill`), Backward Fill (`bfill`), and population median fallback.
5. **Clinical Biomarker & Feature Engineering**:
   - **SIRS Criteria**: Systemic Inflammatory Response Syndrome criteria (Temp, HR, Resp, WBC)
   - **Shock Index**: Calculated as $HR / SBP$
   - **qSOFA Score**: Quick SOFA bedside mortality screening
   - **3-Hour Rolling Vital Trajectories**: Delta changes for Heart Rate ($\Delta HR_{3h}$) and Mean Arterial Pressure ($\Delta MAP_{3h}$)
6. **Patient-Level Stratified Partitioning & Scaling**: Split by unique `Patient_ID` to eliminate temporal data leakage, and fit `RobustScaler`.
7. **Dataset Export**: Save the processed dataset to `physionet_sepsis_processed.csv` and export scaler metadata.

In [ ]:
# Ensure working directory is the project root regardless of how the notebook is opened
import os
import sys

if os.path.basename(os.getcwd()) in ['jupyter_notebooks', 'notebooks']:
    os.chdir('..')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Active Project Root: {os.getcwd()}")

In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
import joblib

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("Libraries imported successfully!")

## Step 1: Raw Data Ingestion & Stitching
The raw PhysioNet 2019 dataset contains thousands of individual pipe-delimited (`.psv`) files (one file per patient stay).  
If `physionet_sepsis_combined.csv` is already generated, we load it directly; otherwise, we stitch all individual `.psv` files from `archive/` into a single combined DataFrame.

In [ ]:
combined_path = 'physionet_sepsis_combined.csv'

if os.path.exists(combined_path):
    print(f"Loading existing combined dataset from: {combined_path}")
    df = pd.read_csv(combined_path)
else:
    print("Stitching raw PSV files from archive directories...")
    path_a = glob.glob('./archive/training_setA/training_setA/*.psv') or glob.glob('./archive/training_setA/*.psv')
    path_b = glob.glob('./archive/training_setB/*.psv')
    all_files = path_a + path_b
    print(f"Found {len(all_files):,} patient files. Starting concatenation...")
    
    dfs = []
    for f in all_files:
        patient_id = os.path.basename(f).split('.')[0]
        temp_df = pd.read_csv(f, sep='|')
        temp_df['Patient_ID'] = patient_id
        dfs.append(temp_df)
    
    df = pd.concat(dfs, ignore_index=True)
    df.to_csv(combined_path, index=False)
    print(f"Combined dataset saved as '{combined_path}'.")

print(f"Total Rows (Hourly ICU Readings): {len(df):,}")
print(f"Unique Patients: {df['Patient_ID'].nunique():,}")
print(f"Total Columns ({len(df.columns)}): {list(df.columns)}")
df.head()

## Step 2: Exploratory Data Analysis (EDA)
### 2.1 Target Class Distribution (`SepsisLabel`)
Sepsis occurs in only a small percentage of ICU hours. In clinical machine learning, addressing severe class imbalance is paramount.

In [ ]:
sepsis_counts = df['SepsisLabel'].value_counts()
sepsis_pct = df['SepsisLabel'].value_counts(normalize=True) * 100

print("Target Class Distribution:")
print(f" - Non-Sepsis (0): {sepsis_counts.get(0, 0):,} ({sepsis_pct.get(0, 0):.2f}%)")
print(f" - Sepsis (1):     {sepsis_counts.get(1, 0):,} ({sepsis_pct.get(1, 0):.2f}%)")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))

sepsis_counts.plot(kind='bar', color=['#2b5c8f', '#d9534f'], ax=ax[0])
ax[0].set_title('Hourly Sepsis Label Frequency', fontsize=12, fontweight='bold')
ax[0].set_xticklabels(['Non-Sepsis (0)', 'Sepsis (1)'], rotation=0)
ax[0].set_ylabel('Observation Count')

ax[1].pie(sepsis_counts, labels=['Non-Sepsis (0)', 'Sepsis (1)'],
          autopct='%1.2f%%', colors=['#2b5c8f', '#d9534f'], explode=[0, 0.18], startangle=90)
ax[1].set_title('Sepsis Label Proportion', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### 2.2 Missing Data Profiling
Clinical EHR records have high rates of missing values. Routine vital signs (Heart Rate, Blood Pressure, Respiration) are continuously measured, while laboratory biomarkers (Lactate, WBC, Creatinine) are measured once or twice daily.

In [ ]:
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)

plt.figure(figsize=(16, 6))
missing_pct.plot(kind='bar', color='#3498db', width=0.8)
plt.title('Percentage of Missing Values per Feature in Raw Dataset', fontsize=14, fontweight='bold')
plt.ylabel('Missing Percentage (%)')
plt.xlabel('Features')
plt.axhline(50, color='orange', linestyle='--', label='50% Missing Threshold')
plt.axhline(90, color='red', linestyle='--', label='90% Missing Threshold')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Features with >90% missing values: {list(missing_pct[missing_pct > 90].index)}")

## Step 3: Outlier Cleaning & Range Validation
We apply physiological bounding to reset impossible readings (e.g. sensor disconnects, noise artifacts) to `NaN` prior to imputation. Any columns with 100% missing values (e.g., `EtCO2`) are safely dropped.

In [ ]:
valid_ranges = {
    'HR': (20, 250),
    'O2Sat': (20, 100),
    'Temp': (25, 43),
    'SBP': (30, 270),
    'MAP': (20, 250),
    'DBP': (20, 200),
    'Resp': (4, 65),
    'pH': (6.5, 8.0),
    'Glucose': (10, 1000),
    'WBC': (0.1, 300),
    'Lactate': (0.1, 35)
}

cleaned_outliers = 0
for col, (min_val, max_val) in valid_ranges.items():
    if col in df.columns:
        invalid_mask = (df[col] < min_val) | (df[col] > max_val)
        cnt = invalid_mask.sum()
        if cnt > 0:
            df.loc[invalid_mask, col] = np.nan
            cleaned_outliers += cnt
            print(f" - {col:10s}: reset {cnt:6,d} out-of-range values [{min_val}, {max_val}] to NaN.")

# Drop columns with 100% missing values
all_null_cols = df.columns[df.isnull().mean() == 1.0].tolist()
if all_null_cols:
    print(f"Dropping 100% missing columns: {all_null_cols}")
    df = df.drop(columns=all_null_cols)

print(f"Outlier cleaning complete. Total invalid measurements reset: {cleaned_outliers:,}")

## Step 4: Patient Longitudinal Imputation
To preserve clinical temporal integrity without introducing lookahead bias:
1. **Patient-level Forward Fill (`ffill`)**: Last Observation Carried Forward (LOCF).
2. **Patient-level Backward Fill (`bfill`)**: Initial readings for early ICU hours before labs return.
3. **Population Median Imputation**: Fallback for remaining unobserved measurements across the entire cohort.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'SepsisLabel' in numeric_cols:
    numeric_cols.remove('SepsisLabel')

print("1. Performing Patient-level Forward Fill (ffill)...")
df[numeric_cols] = df.groupby('Patient_ID')[numeric_cols].ffill()

print("2. Performing Patient-level Backward Fill (bfill)...")
df[numeric_cols] = df.groupby('Patient_ID')[numeric_cols].bfill()

print("3. Imputing remaining NaNs with Population Median...")
pop_medians = df[numeric_cols].median()
df[numeric_cols] = df[numeric_cols].fillna(pop_medians)

print(f"Remaining null values across numeric columns: {df[numeric_cols].isnull().sum().sum()}")

## Step 5: Clinical Biomarker & Feature Engineering
We compute established clinical criteria that clinicians use at the bedside:
- **SIRS Criteria**: Evaluates systemic inflammation across Temperature, Heart Rate, Respiration Rate, and WBC.
- **Shock Index**: Calculated as $\text{Shock Index} = \frac{HR}{SBP}$; values $> 0.7$ indicate occult hemodynamic collapse.
- **qSOFA Score**: Quick SOFA score evaluating bedside sepsis morbidity ($Resp \ge 22$, $SBP \le 100$).
- **3-Hour Rolling Vital Deltas**: Captures sudden deterioration trajectories in Heart Rate and MAP.

In [ ]:
# 5.1 SIRS Criteria Calculation
df['SIRS_Temp'] = ((df['Temp'] > 38.0) | (df['Temp'] < 36.0)).astype(int)
df['SIRS_HR'] = (df['HR'] > 90.0).astype(int)
paco2_ref = df['PaCO2'] if 'PaCO2' in df.columns else pd.Series(40.0, index=df.index)
df['SIRS_Resp'] = ((df['Resp'] > 20.0) | (paco2_ref < 32.0)).astype(int)
df['SIRS_WBC'] = ((df['WBC'] > 12.0) | (df['WBC'] < 4.0)).astype(int)
df['SIRS_Score'] = df['SIRS_Temp'] + df['SIRS_HR'] + df['SIRS_Resp'] + df['SIRS_WBC']
df['SIRS_Positive'] = (df['SIRS_Score'] >= 2).astype(int)

# 5.2 Shock Index (HR / SBP)
df['Shock_Index'] = (df['HR'] / (df['SBP'] + 1e-5)).round(4)

# 5.3 qSOFA Score
df['qSOFA_Resp'] = (df['Resp'] >= 22.0).astype(int)
df['qSOFA_SBP'] = (df['SBP'] <= 100.0).astype(int)
df['qSOFA_Score'] = df['qSOFA_Resp'] + df['qSOFA_SBP']

# 5.4 3-Hour Rolling Trajectory Deltas
df['HR_3h_mean'] = df.groupby('Patient_ID')['HR'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['HR_delta_3h'] = (df['HR'] - df['HR_3h_mean']).round(2)

df['MAP_3h_mean'] = df.groupby('Patient_ID')['MAP'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['MAP_delta_3h'] = (df['MAP'] - df['MAP_3h_mean']).round(2)

df = df.drop(columns=['HR_3h_mean', 'MAP_3h_mean'])

print("Engineered Clinical Features Summary:")
df[['SIRS_Score', 'SIRS_Positive', 'Shock_Index', 'qSOFA_Score', 'HR_delta_3h', 'MAP_delta_3h']].describe()

## Step 6: Patient-Level Stratified Partitioning & Scaling
To ensure **zero data leakage**, the dataset is partitioned **by unique Patient ID** (rather than individual hourly rows).  
We fit a `RobustScaler` (resistant to clinical outliers) strictly on the training partition and serialize it as `sepsis_scaler.pkl`.

In [ ]:
patient_ids = df['Patient_ID'].unique()
patient_sepsis = df.groupby('Patient_ID')['SepsisLabel'].max()

train_pids, temp_pids = train_test_split(
    patient_ids, test_size=0.30, random_state=42, stratify=patient_sepsis[patient_ids]
)
val_pids, test_pids = train_test_split(
    temp_pids, test_size=0.50, random_state=42, stratify=patient_sepsis[temp_pids]
)

print(f"Total Unique Patients: {len(patient_ids):,}")
print(f" - Train Cohort: {len(train_pids):,} patients ({len(train_pids)/len(patient_ids)*100:.1f}%)")
print(f" - Val Cohort:   {len(val_pids):,} patients ({len(val_pids)/len(patient_ids)*100:.1f}%)")
print(f" - Test Cohort:  {len(test_pids):,} patients ({len(test_pids)/len(patient_ids)*100:.1f}%)")

# Fit RobustScaler on training partition
feature_cols = [c for c in df.columns if c not in ['Patient_ID', 'SepsisLabel']]
scaler = RobustScaler()
train_mask = df['Patient_ID'].isin(train_pids)
scaler.fit(df.loc[train_mask, feature_cols])

# Serialize scaler and metadata
joblib.dump(scaler, 'sepsis_scaler.pkl')
with open('feature_columns.json', 'w') as f:
    json.dump({
        'feature_cols': feature_cols,
        'total_features': len(feature_cols),
        'target_col': 'SepsisLabel',
        'id_col': 'Patient_ID'
    }, f, indent=4)

print("RobustScaler saved to 'sepsis_scaler.pkl' and metadata saved to 'feature_columns.json'.")

## Step 7: Saving the Processed Dataset
We persist the clean dataset to `physionet_sepsis_processed.csv` for model training and research benchmarking.

In [ ]:
output_csv_path = 'physionet_sepsis_processed.csv'
df.to_csv(output_csv_path, index=False)
print(f"SUCCESS: Processed dataset saved to '{output_csv_path}'.")
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"File Size: {os.path.getsize(output_csv_path)/(1024*1024):.1f} MB")